# 09. 커스텀 워크플로와 RAG

## 학습 목표

LangGraph `StateGraph`로 커스텀 워크플로를 만들고, RAG 패턴을 구현합니다.

이 노트북에서 다루는 내용:
- `StateGraph`의 기본 구조 (노드, 엣지, 상태)
- 조건부 엣지를 활용한 분기 처리
- `create_agent`를 워크플로 노드로 통합
- RAG (Retrieval-Augmented Generation) 패턴 구현

## 9.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

from langchain.agents import create_agent
from langchain.tools import tool

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 9.2 StateGraph 기초

LangGraph의 핵심 빌딩 블록을 살펴봅니다.

| 개념 | 설명 |
|------|------|
| **노드(Node)** | 처리 단위. 함수 또는 에이전트가 될 수 있습니다 |
| **엣지(Edge)** | 노드 간 연결. 실행 흐름을 정의합니다 |
| **상태(State)** | 노드 간 공유 데이터. `TypedDict`로 정의합니다 |

`StateGraph`는 이 세 가지를 조합하여 복잡한 워크플로를 구성할 수 있게 합니다.

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# 상태 정의
class WorkflowState(TypedDict):
    input: str
    processed: str
    output: str


# 노드 함수 정의
def preprocess(state: WorkflowState) -> dict:
    """
    전처리: 입력 텍스트를 정리합니다.
    """
    return {
        "processed": state["input"].strip().lower()
    }


def analyze(state: WorkflowState) -> dict:
    """
    분석: 처리된 텍스트를 분석합니다.
    """
    text = state["processed"]
    word_count = len(text.split())

    return {
        "output": f"분석 완료: '{text}'는 {word_count}단어입니다."
    }


# 그래프 구성
builder = StateGraph(WorkflowState)

builder.add_node("preprocess", preprocess)
builder.add_node("analyze", analyze)

builder.add_edge(START, "preprocess")
builder.add_edge("preprocess", "analyze")
builder.add_edge("analyze", END)

graph = builder.compile()


# 실행
result = graph.invoke(
    {
        "input": "  Hello World from LangGraph!  "
    },
    config=lf_config
)

print("입력:", result["input"])
print("전처리:", result["processed"])
print("출력:", result["output"])

입력:   Hello World from LangGraph!  
전처리: hello world from langgraph!
출력: 분석 완료: 'hello world from langgraph!'는 4단어입니다.


## 9.3 조건부 엣지

상태에 따라 다른 경로로 분기합니다. `add_conditional_edges`를 사용하면 런타임에 동적으로 다음 노드를 선택할 수 있습니다.

아래 예제에서는 입력 텍스트를 분류한 후, 카테고리에 따라 서로 다른 핸들러로 라우팅합니다.

In [4]:
class RouterState(TypedDict):
    input: str
    category: str
    output: str

def classify(state: RouterState) -> dict:
    """입력을 분류합니다."""
    text = state["input"].lower()
    if any(w in text for w in ["calculate", "math", "sum"]):
        return {"category": "math"}
    elif any(w in text for w in ["translate", "language"]):
        return {"category": "language"}
    return {"category": "general"}

def handle_math(state: RouterState) -> dict:
    return {"output": f"[수학 핸들러] 처리 중: {state['input']}"}

def handle_language(state: RouterState) -> dict:
    return {"output": f"[언어 핸들러] 처리 중: {state['input']}"}

def handle_general(state: RouterState) -> dict:
    return {"output": f"[일반 핸들러] 처리 중: {state['input']}"}

def route(state: RouterState) -> str:
    """분류 결과에 따라 라우팅"""
    return state["category"]

builder = StateGraph(RouterState)
builder.add_node("classify", classify)
builder.add_node("math", handle_math)
builder.add_node("language", handle_language)
builder.add_node("general", handle_general)

builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, {
    "math": "math",
    "language": "language",
    "general": "general",
})
builder.add_edge("math", END)
builder.add_edge("language", END)
builder.add_edge("general", END)

graph = builder.compile()

# 테스트
for query in ["2+2를 계산해줘", "hello를 한국어로 번역해줘", "AI란 무엇인가요?"]:
    result = graph.invoke({"input": query}, config=lf_config)
    print(f"  {query} → {result['output']}")

  2+2를 계산해줘 → [일반 핸들러] 처리 중: 2+2를 계산해줘
  hello를 한국어로 번역해줘 → [일반 핸들러] 처리 중: hello를 한국어로 번역해줘
  AI란 무엇인가요? → [일반 핸들러] 처리 중: AI란 무엇인가요?


## 9.4 에이전트를 워크플로에 통합

`create_agent`로 만든 에이전트를 `StateGraph`의 노드로 사용합니다. 이렇게 하면 여러 에이전트를 파이프라인으로 연결하여 복잡한 작업을 처리할 수 있습니다.

아래 예제에서는 리서치 에이전트와 작성 에이전트를 순차적으로 연결합니다.

In [5]:
from langgraph.graph import MessagesState

@tool
def web_search(query: str) -> str:
    """웹에서 정보를 검색합니다."""
    return f"'{query}'에 대한 검색 결과"

# 에이전트를 노드로 사용
research_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="당신은 리서치 어시스턴트입니다. 정보를 검색하고 간결한 답변을 제공하세요.",
    name="researcher",
)

@tool
def format_report(content: str) -> str:
    """콘텐츠를 구조화된 보고서 형식으로 변환합니다."""
    return f"## 보고서\n\n{content}"

writer_agent = create_agent(
    model=model,
    tools=[format_report],
    system_prompt="당신은 테크니컬 라이터입니다. 조사 결과를 깔끔한 보고서로 정리하세요.",
    name="writer",
)

# 리서치 → 작성 파이프라인
builder = StateGraph(MessagesState)
builder.add_node(research_agent)
builder.add_node(writer_agent)
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", END)

pipeline = builder.compile()

result = pipeline.invoke(
    {"messages": [{"role": "user", "content": "LangChain v1의 주요 기능을 조사해 주세요."}]},
    config=lf_config,
)
print("파이프라인 결과:", result["messages"][-1].content[:300])

파이프라인 결과: ## 보고서

### LangChain v1의 주요 기능

1. **체인(Chains)**  
   여러 LLM(대형 언어 모델) 호출과 도구 실행을 단계별로 연결해 복잡한 태스크를 수행할 수 있는 기능입니다.

2. **에이전트(Agents)**  
   자연어 지시문에 따라 필요한 도구(검색, 계산 등)를 선택해 동적으로 사용할 수 있습니다.

3. **프롬프트 템플릿(Prompt Template)**  
   다양한 입력을 받아 유연하게 프롬프트를 생성하고 관리할 수 있습니다.

4. **메모리(Memory)**  
   대화


## 9.5 RAG (Retrieval-Augmented Generation) 개요

RAG는 외부 지식을 검색하여 LLM의 응답을 보강하는 패턴입니다. 3가지 주요 접근 방식이 있습니다:

| 패턴 | 설명 | 특징 |
|------|------|------|
| **기본 2단계** | 검색 → 생성 | 단순하고 빠름 |
| **에이전틱 RAG** | 에이전트가 검색 도구를 호출하여 반복 | 유연하고 정확 |
| **하이브리드** | 키워드 + 시맨틱 검색 결합 | 검색 품질 향상 |

- **기본 2단계**: 쿼리로 문서를 검색한 후, 검색된 문서를 컨텍스트로 LLM에 전달하여 답변을 생성합니다.
- **에이전틱 RAG**: 에이전트가 검색 도구를 사용하여 필요한 정보를 반복적으로 검색하고, 충분한 정보를 모은 후 답변합니다.

## 9.6 간단한 RAG 구현

텍스트를 청크로 분할하고, 간단한 키워드 기반 검색으로 RAG를 구현합니다. 벡터 스토어 없이도 RAG의 핵심 개념을 이해할 수 있습니다.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 샘플 문서
documents = [
    "LangChain is a framework for building applications with large language models. It provides tools for prompt engineering, memory management, and agent creation.",
    "LangGraph is a low-level orchestration framework for building stateful agents. It uses a graph-based approach with nodes and edges.",
    "Middleware in LangChain v1 allows you to intercept and modify agent behavior at every step. You can add logging, guardrails, and custom logic.",
    "Multi-agent systems in LangChain support five patterns: subagents, handoffs, skills, router, and custom workflows.",
    "RAG (Retrieval-Augmented Generation) combines information retrieval with text generation to provide grounded, factual responses.",
]

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
)

chunks = []
for doc in documents:
    chunks.extend(text_splitter.split_text(doc))

print(f"원본 문서: {len(documents)}개")
print(f"분할 청크: {len(chunks)}개")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] {chunk[:80]}...")

원본 문서: 5개
분할 청크: 5개
  [0] LangChain is a framework for building applications with large language models. I...
  [1] LangGraph is a low-level orchestration framework for building stateful agents. I...
  [2] Middleware in LangChain v1 allows you to intercept and modify agent behavior at ...
  [3] Multi-agent systems in LangChain support five patterns: subagents, handoffs, ski...
  [4] RAG (Retrieval-Augmented Generation) combines information retrieval with text ge...


In [7]:
# 간단한 키워드 기반 검색 (벡터 스토어 없이)
def simple_search(query: str, documents: list[str], top_k: int = 2) -> list[str]:
    """간단한 키워드 기반 검색입니다."""
    scores = []
    query_words = set(query.lower().split())
    for doc in documents:
        doc_words = set(doc.lower().split())
        score = len(query_words & doc_words)
        scores.append((score, doc))
    scores.sort(reverse=True)
    return [doc for _, doc in scores[:top_k]]

# RAG 도구
@tool
def retrieve_info(query: str) -> str:
    """지식 베이스에서 관련 정보를 검색합니다."""
    results = simple_search(query, chunks, top_k=3)
    return "\n\n".join(results)

# RAG 에이전트
rag_agent = create_agent(
    model=model,
    tools=[retrieve_info],
    system_prompt="""당신은 지식이 풍부한 어시스턴트입니다. 항상 retrieve_info 도구를 사용하여 정보를 검색한 후 답변하세요.
검색된 정보를 바탕으로 답변하세요.""",
)

result = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "LangChain의 멀티에이전트 패턴은 무엇인가요?"}]},
    config=lf_config,
)
print("RAG 에이전트 결과:", result["messages"][-1].content)

RAG 에이전트 결과: LangChain의 멀티에이전트 패턴은 여러 에이전트가 협력하거나 역할을 분담하여 문제를 해결하는 구조를 말합니다. LangChain은 대표적으로 다음과 같은 다섯 가지 멀티에이전트 패턴을 지원합니다:

1. Subagents: 메인 에이전트가 하위 에이전트(서브에이전트)들을 호출해 특정 작업을 분담합니다.
2. Handoffs: 한 에이전트가 작업을 처리하다가 다른 에이전트에게 작업을 넘기는 구조입니다.
3. Skills: 여러 에이전트가 각각의 전문성을 바탕으로 특정 “스킬(기능)”을 담당합니다.
4. Router: 입력을 보고 해당 작업에 적합한 에이전트에게 라우팅(분배)합니다.
5. Custom workflows: 개발자가 직접 원하는 단계와 에이전트 조합을 설계하여 워크플로를 만듭니다.

이런 패턴을 통해 하나의 에이전트로 처리하기 어려운 복잡한 문제도 효과적으로 자동화 및 해결할 수 있습니다. LangChain v1에서는 에이전트의 동작을 중간에서 제어·수정할 수 있는 미들웨어도 지원하여 유연하게 멀티에이전트 시스템을 구축할 수 있습니다.


## 9.7 FAISS 벡터 스토어 (선택)

임베딩 기반 유사도 검색을 사용하면 키워드 매칭보다 훨씬 정확한 검색이 가능합니다. 아래는 FAISS 벡터 스토어를 사용하는 예시입니다.

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 임베딩 모델  생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 스토어 생성
vectorstore = FAISS.from_texts(chunks, embeddings)

# 검색
results = vectorstore.similarity_search("LangChain agent patterns", k=3)
for doc in results:
    print(doc.page_content)

Multi-agent systems in LangChain support five patterns: subagents, handoffs, skills, router, and custom workflows.
LangChain is a framework for building applications with large language models. It provides tools for prompt engineering, memory management, and agent creation.
Middleware in LangChain v1 allows you to intercept and modify agent behavior at every step. You can add logging, guardrails, and custom logic.


## 9.8 요약

이 노트북에서 배운 내용:

| 주제 | 핵심 내용 |
|------|----------|
| **StateGraph 기초** | 노드, 엣지, 상태로 워크플로를 구성합니다 |
| **조건부 엣지** | `add_conditional_edges`로 런타임에 분기 처리합니다 |
| **에이전트 통합** | `create_agent`를 StateGraph 노드로 사용하여 파이프라인을 구성합니다 |
| **RAG 패턴** | 검색 + 생성을 결합하여 사실 기반 응답을 제공합니다 |
| **벡터 스토어** | FAISS 등으로 임베딩 기반 유사도 검색이 가능합니다 |

다음 노트북에서는 에이전트를 프로덕션 환경으로 배포하는 방법을 알아봅니다.

### 다음 단계
→ **[10_production.ipynb](./10_production.ipynb)**: 프로덕션 배포를 배웁니다.
